In [1]:
# Computer vision
import cv2
import mediapipe as mp

# Audio playback
import pygame

# Utilities
import time
import numpy as np

print("✅ Cell 1: Libraries imported successfully")


pygame 2.6.1 (SDL 2.28.4, Python 3.9.25)
Hello from the pygame community. https://www.pygame.org/contribute.html
✅ Cell 1: Libraries imported successfully


C:\Users\utkar\anaconda3\envs\mp_solutions\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=1
)

print("✅ Cell 2: MediaPipe hand model initialized")


✅ Cell 2: MediaPipe hand model initialized


In [3]:
cap = cv2.VideoCapture(0)

if cap.isOpened():
    print("✅ Cell 3: Webcam accessed successfully")
else:
    print("❌ Webcam not detected")


✅ Cell 3: Webcam accessed successfully


In [4]:
pygame.mixer.init()

disco = pygame.mixer.Sound("sounds/disco.wav")
clap = pygame.mixer.Sound("sounds/clap.wav")
amazing = pygame.mixer.Sound("sounds/amazing.wav")

print("✅ Cell 4: Audio system initialized & sounds loaded")


✅ Cell 4: Audio system initialized & sounds loaded


In [5]:
current_sound = None     # currently playing sound
last_gesture = None     # last valid gesture trigger
cooldown = 1.0          # seconds
last_time = 0
DEBUG = True

print("✅ Cell 5: Control variables initialized")


✅ Cell 5: Control variables initialized


In [6]:
def get_gesture(lm):
    fingers = []
    
    fingers.append(lm[8].y < lm[6].y)    # Index
    fingers.append(lm[12].y < lm[10].y)  # Middle
    fingers.append(lm[16].y < lm[14].y)  # Ring
    fingers.append(lm[20].y < lm[18].y)  # Pinky

    if sum(fingers) == 0:
        return "FIST"
    elif fingers == [1,0,0,0]:
        return "ONE"
    elif fingers == [1,1,0,0]:
        return "TWO"
    elif sum(fingers) == 4:
        return "PALM"
    else:
        return "UNKNOWN"

print("✅ Cell 6: Gesture detection logic ready")


✅ Cell 6: Gesture detection logic ready


In [7]:
print("🚀 Gesture DJ running — Press 'q' to exit")

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Camera frame not received")
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    gesture = None
    status_text = "STOPPED"

    if result.multi_hand_landmarks:
        for hand_lms in result.multi_hand_landmarks:
            lm = hand_lms.landmark
            gesture = get_gesture(lm)
            current_time = time.time()

            # ✊ STOP — ONLY WHEN FIST
            if gesture == "FIST":
                if current_sound:
                    current_sound.stop()
                    current_sound = None
                    if DEBUG:
                        print("✊ FIST → STOP (explicit)")
                last_gesture = "FIST"

            # ☝️ ONE → DISCO (trigger once, not hold)
            elif gesture == "ONE" and last_gesture != "ONE" and (current_time - last_time) > cooldown:
                if current_sound:
                    current_sound.stop()
                disco.play()
                current_sound = disco
                last_gesture = "ONE"
                last_time = current_time
                if DEBUG:
                    print("☝️ ONE → disco.wav started")

            # ✋ PALM → CLAP
            elif gesture == "PALM" and last_gesture != "PALM" and (current_time - last_time) > cooldown:
                if current_sound:
                    current_sound.stop()
                clap.play()
                current_sound = clap
                last_gesture = "PALM"
                last_time = current_time
                if DEBUG:
                    print("✋ PALM → clap.wav started")

            # ✌️ TWO → AMAZING
            elif gesture == "TWO" and last_gesture != "TWO" and (current_time - last_time) > cooldown:
                if current_sound:
                    current_sound.stop()
                amazing.play()
                current_sound = amazing
                last_gesture = "TWO"
                last_time = current_time
                if DEBUG:
                    print("✌️ TWO → amazing.wav started")

            # Draw landmarks
            mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

    # -------- STATUS TEXT --------
    if current_sound == disco:
        status_text = "DISCO PLAYING"
    elif current_sound == clap:
        status_text = "CLAP PLAYING"
    elif current_sound == amazing:
        status_text = "AMAZING PLAYING"

    cv2.putText(frame, f"Gesture: {gesture}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, status_text, (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

    cv2.putText(frame, "Press 'q' to exit", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Gesture Controlled DJ", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("🛑 Program exited by user (q pressed)")
        break

cap.release()
cv2.destroyAllWindows()


🚀 Gesture DJ running — Press 'q' to exit


C:\Users\utkar\anaconda3\envs\mp_solutions\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


☝️ ONE → disco.wav started
✊ FIST → STOP (explicit)
✌️ TWO → amazing.wav started
✊ FIST → STOP (explicit)
☝️ ONE → disco.wav started
✊ FIST → STOP (explicit)
☝️ ONE → disco.wav started
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
✌️ TWO → amazing.wav started
✊ FIST → STOP (explicit)
☝️ ONE → disco.wav started
✊ FIST → STOP (explicit)
☝️ ONE → disco.wav started
✋ PALM → clap.wav started
✊ FIST → STOP (explicit)
🛑 Program exited by user (q pressed)
